In [1]:
import csv
import requests

import numpy as np
import pandas as pd
import ast

## PTB XL - Dataset Exploration
References: http://webimatics.univ-lyon1.fr/scp-ecg/, https://www.nature.com/articles/s41597-020-0495-6#ref-CR3

### Classes and Categories

There are **71** unique SCP-ECG statements used in the dataset. We categorize them by assigning each statement to one or more of the following categories: **diagnostic, form and rhythm** statements. There are **44 different diagnostic** statements, **19 different form** statements describing the form of the ECG signal, where 4 statements for diagnostic and form coincide, **12 different non-overlapping rhythm statements** describing the cardiac rhythm

| <img src="https://media.springernature.com/lw685/springer-static/image/art%3A10.1038%2Fs41597-020-0495-6/MediaObjects/41597_2020_495_Fig1_HTML.png" alt="Classes" style="width: 300px;"/> | <img src="https://media.springernature.com/lw685/springer-static/image/art%3A10.1038%2Fs41597-020-0495-6/MediaObjects/41597_2020_495_Fig4_HTML.png" alt="Categories" style="width: 300px;"/>   |
|------|------|

### Parent Classes (Class Hierarchy)
For diagnostic statements we provide a hierarchy of superclasses and subclasses that can be used to train classification algorithms on a set of broader categories instead of the original fine-grained diagnostic labels

### Likelihood

The likelihood ranges from 0 to 100 conveying the certainty the cardiologist (if the diagnosing cardiologist is very certain about a statement). For form and rhythm statements or in cases where no likelihood information was available, the corresponding likelihood was set to zero. 

### Diagnostic/Form/Rhythm Classes
- Diagnostic: https://www.nature.com/articles/s41597-020-0495-6/tables/7
- Form: https://www.nature.com/articles/s41597-020-0495-6/tables/8
- Rhythm: https://www.nature.com/articles/s41597-020-0495-6/tables/9


In [2]:
# PTB-XL Dataset
ptb_dataset_link = "https://www.physionet.org/files/ptb-xl/1.0.1/ptbxl_database.csv?download"
# PTB-XL SCP Classes
ptb_classes_link = "https://www.physionet.org/files/ptb-xl/1.0.1/scp_statements.csv"

In [3]:
# Get all classes from ptb_database
ptb_classes = []
with requests.Session() as s:
    download = s.get(ptb_classes_link)
    decoded_content = download.content.decode('utf-8')
    cr = csv.reader(decoded_content.splitlines(), delimiter=',')
    my_list = list(cr)
    for row in my_list:
        ptb_classes.append(row)

In [4]:
all_ptb_short = []
diagnostic_ptb_short = []
form_ptb_short = []
rhythm_ptb_short = []
#print(ptb_classes)
for ptb_clss in ptb_classes[1:]:
    all_ptb_short.append(ptb_clss[0])
    if ptb_clss[2] == '1.0':
        diagnostic_ptb_short.append(ptb_clss[0])
    if ptb_clss[3] == '1.0':
        form_ptb_short.append(ptb_clss[0])
    if ptb_clss[4] == '1.0':
        rhythm_ptb_short.append(ptb_clss[0])

In [5]:
print('PTB has', len(all_ptb_short), 'classes'\
      '( DIAGNOSTIC [', len(diagnostic_ptb_short),
      '] FORM [',len(form_ptb_short),
      '] RHYTHM [', len(rhythm_ptb_short), '] )')

PTB has 71 classes( DIAGNOSTIC [ 44 ] FORM [ 19 ] RHYTHM [ 12 ] )


In [6]:
# What are form and rhythm classes???
print(form_ptb_short)

['NDT', 'NST_', 'DIG', 'LNGQT', 'ABQRS', 'PVC', 'STD_', 'VCLVH', 'QWAVE', 'LOWT', 'NT_', 'PAC', 'LPR', 'INVT', 'LVOLT', 'HVOLT', 'TAB_', 'STE_', 'PRC(S)']


In [7]:
# 	# Records	Description	Superclass	Subclass
diagnostic = """LAFB	1626	left anterior fascicular block	CD	LAFB/LPFB
IRBBB	1118	incomplete right bundle branch block	CD	IRBBB
AVB	797	first degree AV block	CD	_AVB
IVCD	789	non-specific intraventricular conduction disturbance (block)	CD	IVCD
CRBBB	542	complete right bundle branch block	CD	CRBBB
CLBBB	536	complete left bundle branch block	CD	CLBBB
LPFB	177	left posterior fascicular block	CD	LAFB/LPFB
WPW	80	Wolff-Parkinson-White syndrome	CD	WPW
ILBBB	77	incomplete left bundle branch block	CD	ILBBB
3AVB	16	third degree AV block	CD	_AVB
2AVB	14	second degree AV block	CD	_AVB
LVH	2137	left ventricular hypertrophy	HYP	LVH
LAO/LAE	427	left atrial overload/enlargement	HYP	LAO/LAE
RVH	126	right ventricular hypertrophy	HYP	RVH
RAO/RAE	99	right atrial overload/enlargement	HYP	RAO/RAE
SEHYP	30	septal hypertrophy	HYP	SEHYP
IMI	2685	inferior myocardial infarction	MI	IMI
ASMI	2363	anteroseptal myocardial infarction	MI	AMI
ILMI	479	inferolateral myocardial infarction	MI	IMI
AMI	354	anterior myocardial infarction	MI	AMI
ALMI	290	anterolateral myocardial infarction	MI	AMI
INJAS	215	subendocardial injury in anteroseptal leads	MI	AMI
LMI	201	lateral myocardial infarction	MI	LMI
INJAL	148	subendocardial injury in anterolateral leads	MI	AMI
IPLMI	51	inferoposterolateral myocardial infarction	MI	IMI
IPMI	33	inferoposterior myocardial infarction	MI	IMI
INJIN	18	subendocardial injury in inferior leads	MI	IMI
PMI	17	posterior myocardial infarction	MI	PMI
INJLA	17	subendocardial injury in lateral leads	MI	AMI
INJIL	15	subendocardial injury in inferolateral leads	MI	IMI
NORM	9528	normal ECG	NORM	NORM
NDT	1829	non-diagnostic T abnormalities	STTC	STTC
NST_	770	non-specific ST changes	STTC	NST_
DIG	181	digitalis-effect	STTC	STTC
LNGQT	118	long QT-interval	STTC	STTC
ISC_	1275	non-specific ischemic	STTC	ISC_
ISCAL	660	ischemic in anterolateral leads	STTC	ISCA
ISCIN	219	ischemic in inferior leads	STTC	ISCI
ISCIL	179	ischemic in inferolateral leads	STTC	ISCI
ISCAS	170	ischemic in anteroseptal leads	STTC	ISCA
ISCLA	142	ischemic in lateral leads	STTC	ISCA
ANEUR	104	ST-T changes compatible with ventricular aneurysm	STTC	STTC
EL	97	electrolytic disturbance or drug (former EDIS)	STTC	STTC
ISCAN	44	ischemic in anterior leads	STTC	ISCA"""
# # Records	Description
form = 	"""NDT	1829	non-diagnostic T abnormalities
NST_	770	non-specific ST changes
DIG	181	digitalis-effect
LNGQT	118	long QT-interval
ABQRS	3327	abnormal QRS
PVC	1146	ventricular premature complex
STD_	1009	non-specific ST depression
VCLVH	875	voltage criteria (QRS) for left ventricular hypertrophy
QWAVE	548	Q waves present
LOWT	438	low amplitude T-waves
NT_	424	non-specific T-wave changes
PAC	398	atrial premature complex
LPR	340	prolonged PR interval
INVT	294	inverted T-waves
LVOLT	182	low QRS voltages in the frontal and horizontal leads
HVOLT	62	high QRS voltage
TAB_	35	T-wave abnormality
STE_	28	non-specific ST elevation
PRC(S)	10	premature complex(es)"""
# # Records	Description
rhythm ="""SR	16782	sinus rhythm
AFIB	1514	atrial fibrillation
STACH	826	sinus tachycardia
SARRH	772	sinus arrhythmia
SBRAD	637	sinus bradycardia
PACE	296	normal functioning artificial pacemaker
SVARR	157	supraventricular arrhythmia
BIGU	82	bigeminal pattern (unknown origin, SV or Ventricular)
AFLT	73	atrial flutter
SVTAC	27	supraventricular tachycardia
PSVT	24	paroxysmal supraventricular tachycardia
TRIGU	20	trigeminal pattern (unknown origin, SV or Ventricular)"""

diagnostic_superclasses="""NORM	Normal ECG
CD	Conduction Disturbance
MI	Myocardial Infarction
HYP	Hypertrophy
STTC	ST/T change"""
diagnostic_subclasses="""NORM	NORM	Normal ECG
CD	LAFB/LPFB	left anterior/left posterior fascicular block
CD	IRBBB	incomplete right bundle branch block
CD	ILBBB	incomplete left bundle branch block
CD	CLBBB	complete left bundle branch block
CD	CRBBB	complete right bundle branch block
CD	_AVB	AV block
CD	IVCB	non-specific intraventricular conduction disturbance (block)
CD	WPW	Wolff-Parkinson-White syndrome
HYP	LVH	left ventricular hypertrophy
HYP	RHV	right ventricular hypertrophy
HYP	LAO/LAE	left atrial overload/enlargement
HYP	RAO/RAE	right atrial overload/enlargement
HYP	SEHYP	septal hypertrophy
MI	AMI	anterior myocardial infarction
MI	IMI	inferior myocardial infarction
MI	LMI	lateral myocardial infarction
MI	PMI	posterior myocardial infarction
STTC	ISCA	ischemic in anterior leads
STTC	ISCI	ischemic in inferior leads
STTC	ISC_	non-specific ischemic
STTC	STTC	ST-T changes
STTC	NST_	non-specific ST changes"""
heart_angle="""UNK	Unknown	8505
MID	Normal axis	7687
LAD	Left axis deviation	3764
ALAD	Abnormal LAD, extreme left axis deviation	1382
RAD	Right axis deviation	221
ARAD	Abnormal RAD, extreme right axis deviation	122
AXL	Horizontal axis	102
AXR	Vertical axis	51
SAG	Saggital type (S1-S2-S3 Pattern)	3"""

diagnostic_superclasses = [r.split('\t') for r in diagnostic_superclasses.split('\n')]
diagnostic_subclasses = [r.split('\t') for r in diagnostic_subclasses.split('\n')]
diagnostic_labels = [[r.split('\t')[0], r.split('\t')[-1], r.split('\t')[-2], r.split('\t')[-3]]
                     for r in diagnostic.split('\n')]
form = [[r.split('\t')[0], r.split('\t')[2]] for r in form.split('\n')]
rhythm = [[r.split('\t')[0], r.split('\t')[2]] for r in rhythm.split('\n')]
heart_angle = [[r.split('\t')[0], r.split('\t')[1]] for r in heart_angle.split('\n')]

In [8]:
display("FORM", form, "RYTHM", rhythm, "ANGLE", heart_angle, "DIAGNOSTIC", diagnostic_labels)

'FORM'

[['NDT', 'non-diagnostic T abnormalities'],
 ['NST_', 'non-specific ST changes'],
 ['DIG', 'digitalis-effect'],
 ['LNGQT', 'long QT-interval'],
 ['ABQRS', 'abnormal QRS'],
 ['PVC', 'ventricular premature complex'],
 ['STD_', 'non-specific ST depression'],
 ['VCLVH', 'voltage criteria (QRS) for left ventricular hypertrophy'],
 ['QWAVE', 'Q waves present'],
 ['LOWT', 'low amplitude T-waves'],
 ['NT_', 'non-specific T-wave changes'],
 ['PAC', 'atrial premature complex'],
 ['LPR', 'prolonged PR interval'],
 ['INVT', 'inverted T-waves'],
 ['LVOLT', 'low QRS voltages in the frontal and horizontal leads'],
 ['HVOLT', 'high QRS voltage'],
 ['TAB_', 'T-wave abnormality'],
 ['STE_', 'non-specific ST elevation'],
 ['PRC(S)', 'premature complex(es)']]

'RYTHM'

[['SR', 'sinus rhythm'],
 ['AFIB', 'atrial fibrillation'],
 ['STACH', 'sinus tachycardia'],
 ['SARRH', 'sinus arrhythmia'],
 ['SBRAD', 'sinus bradycardia'],
 ['PACE', 'normal functioning artificial pacemaker'],
 ['SVARR', 'supraventricular arrhythmia'],
 ['BIGU', 'bigeminal pattern (unknown origin, SV or Ventricular)'],
 ['AFLT', 'atrial flutter'],
 ['SVTAC', 'supraventricular tachycardia'],
 ['PSVT', 'paroxysmal supraventricular tachycardia'],
 ['TRIGU', 'trigeminal pattern (unknown origin, SV or Ventricular)']]

'ANGLE'

[['UNK', 'Unknown'],
 ['MID', 'Normal axis'],
 ['LAD', 'Left axis deviation'],
 ['ALAD', 'Abnormal LAD, extreme left axis deviation'],
 ['RAD', 'Right axis deviation'],
 ['ARAD', 'Abnormal RAD, extreme right axis deviation'],
 ['AXL', 'Horizontal axis'],
 ['AXR', 'Vertical axis'],
 ['SAG', 'Saggital type (S1-S2-S3 Pattern)']]

'DIAGNOSTIC'

[['LAFB', 'LAFB/LPFB', 'CD', 'left anterior fascicular block'],
 ['IRBBB', 'IRBBB', 'CD', 'incomplete right bundle branch block'],
 ['AVB', '_AVB', 'CD', 'first degree AV block'],
 ['IVCD',
  'IVCD',
  'CD',
  'non-specific intraventricular conduction disturbance (block)'],
 ['CRBBB', 'CRBBB', 'CD', 'complete right bundle branch block'],
 ['CLBBB', 'CLBBB', 'CD', 'complete left bundle branch block'],
 ['LPFB', 'LAFB/LPFB', 'CD', 'left posterior fascicular block'],
 ['WPW', 'WPW', 'CD', 'Wolff-Parkinson-White syndrome'],
 ['ILBBB', 'ILBBB', 'CD', 'incomplete left bundle branch block'],
 ['3AVB', '_AVB', 'CD', 'third degree AV block'],
 ['2AVB', '_AVB', 'CD', 'second degree AV block'],
 ['LVH', 'LVH', 'HYP', 'left ventricular hypertrophy'],
 ['LAO/LAE', 'LAO/LAE', 'HYP', 'left atrial overload/enlargement'],
 ['RVH', 'RVH', 'HYP', 'right ventricular hypertrophy'],
 ['RAO/RAE', 'RAO/RAE', 'HYP', 'right atrial overload/enlargement'],
 ['SEHYP', 'SEHYP', 'HYP', 'septal hypertrophy'],
 ['IMI'

In [9]:
# Download the original physionet classes file
ptb_info_file = "https://www.physionet.org/files/ptb-xl/1.0.1/ptbxl_database.csv?download"
with requests.Session() as s:
    download = s.get(ptb_info_file)
    decoded_content = download.content.decode('utf-8')
    cr = csv.reader(decoded_content.splitlines(), delimiter=',')
    ptb_info = list(cr)

In [10]:
columns = ptb_info[0]
data = []
rows = []
for row in ptb_info[1:]:
    data.append(row[1:])
    rows.append(row[0])
ptb_info_df = pd.DataFrame(data, index=rows, columns=columns[1:])

ptb_info_df

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
1,15709.0,56.0,1,,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,True,,", I-V1,",,,,,3,records100/00000/00001_lr,records500/00000/00001_hr
2,13243.0,19.0,0,,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,True,,,,,,,2,records100/00000/00002_lr,records500/00000/00002_hr
3,20372.0,37.0,1,,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,True,,,,,,,5,records100/00000/00003_lr,records500/00000/00003_hr
4,17014.0,24.0,0,,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,True,", II,III,AVF",,,,,,3,records100/00000/00004_lr,records500/00000/00004_hr
5,17448.0,19.0,1,,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,True,", III,AVR,AVF",,,,,,4,records100/00000/00005_lr,records500/00000/00005_hr
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21833,17180.0,67.0,1,,,1.0,2.0,AT-60 3,2001-05-31 09:14:35,ventrikulÄre extrasystole(n) sinustachykardie ...,...,True,,", alles,",,,1ES,,7,records100/21000/21833_lr,records500/21000/21833_hr
21834,20703.0,93.0,0,,,1.0,2.0,AT-60 3,2001-06-05 11:33:39,sinusrhythmus lagetyp normal qrs(t) abnorm ...,...,True,,,,,,,4,records100/21000/21834_lr,records500/21000/21834_hr
21835,19311.0,59.0,1,,,1.0,2.0,AT-60 3,2001-06-08 10:30:27,sinusrhythmus lagetyp normal t abnorm in anter...,...,True,,", I-AVR,",,,,,2,records100/21000/21835_lr,records500/21000/21835_hr
21836,8873.0,64.0,1,,,1.0,2.0,AT-60 3,2001-06-09 18:21:49,supraventrikulÄre extrasystole(n) sinusrhythmu...,...,True,,,,,SVES,,8,records100/21000/21836_lr,records500/21000/21836_hr


In [11]:
# Now we want to get the ptb class names to the snomed ids 

### Here is some strange stuff:

# To summarize this long email - it is so important that you go back to the source data and look
# at how it is generated, and be appropriately skeptical of any dataset you are given. 
# In this case you have misinterpreted 0.0 as zero likelihood - which is not what the documentation
# of that data indicates. (Yes - NaN should have been used, instead of 0.0, but that was not our choice
# - we did not create that dataset). But let's back up to the larger question ...

# --> SR is always zero

# The likelihood ranges from 0 to 100 conveying the certainty the cardiologist (if the diagnosing
# cardiologist is very certain about a statement). For form and rhythm statements or in cases where
# no likelihood information was available, the corresponding likelihood was set to zero.

# Subject ID: 77 has a report of "sinusrhythmus unvollständiger rechtsschenkelblock", which translates
# as "sinus rhythm incomplete right bundle branch block", and the SCP Code = {'AMI': 50.0, 'IRBBB':
# 100.0, 'SR': 0.0}.  You can certainly be in sinus rhythm and also have an incomplete RBBB.

## Physionet 2020 Challenge Classes (DX_MAP)

### Organizer Description
Each ECG recording has one or more labels from different type of abnormalities in SNOMED-CT codes. The full list of diagnoses for the challenge has been posted here as a 3 column CSV file: Long-form description, corresponding SNOMED-CT code, abbreviation. Although these descriptions apply to all training data there may be fewer classes in the test data, and in different proportions. However, every class in the test data will be represented in the training data.

Source: http://bioportal.bioontology.org/ontologies/SNOMEDCT

In [12]:
dx_map_link_all = "https://raw.githubusercontent.com/physionetchallenges/" \
    "physionetchallenges.github.io/master/2020/Dx_map.csv"

dx_map_link_scored = "https://raw.githubusercontent.com/physionetchallenges/" \
    "evaluation-2020/master/dx_mapping_scored.csv"

dx_map_link_unscored = "https://raw.githubusercontent.com/physionetchallenges/" \
    "evaluation-2020/master/dx_mapping_unscored.csv"

In [13]:
def get_dx_map_classes(link):
    dx_map_classes = []
    with requests.Session() as s:
        download = s.get(link)
        decoded_content = download.content.decode('utf-8')
        cr = csv.reader(decoded_content.splitlines(), delimiter=',')
        dx_map_classes = list(cr)
    return dx_map_classes

dx_map_all = get_dx_map_classes(dx_map_link_all)
dx_map_scored = get_dx_map_classes(dx_map_link_scored)
dx_map_unscored = get_dx_map_classes(dx_map_link_unscored)

In [14]:
all_dx_map_short = []
for dx_map_clss in dx_map_all[1:]:
    all_dx_map_short.append(dx_map_clss[-1])

scored_dx_map_short = []
for dx_map_clss in dx_map_scored[1:]:
    scored_dx_map_short.append(dx_map_clss[2])
    
unscored_dx_map_short = []
for dx_map_clss in dx_map_unscored[1:]:
    unscored_dx_map_short.append(dx_map_clss[2])

In [15]:
print('Challenge has', len(all_dx_map_short),
      'classes, with scored:', len(scored_dx_map_short), "and unscored:", len(unscored_dx_map_short))

Challenge has 111 classes, with scored: 27 and unscored: 84


## SNOMED-CT Names/Classes

There are some strange terms using the snomed system. For example they differentiate between:
- (finding): Clinical finding: normal/abnormal observations, judgments, or assessments of patients
- (disorder): Disorder: always and necessarily an abnormal clinical state

What does this mean for our classes in the challenge? - I do not know.

In [16]:
# Let's take a look all the snomed ids
# Read all the header files and get all snomeds
import sys
sys.path.append('../utils/')
from import_challenge2020 import get_meta_from_header
import pathlib as pl

clss_someds = []
for file in pl.Path('../utils/temp/').iterdir():
    if file.name.split('.')[-1] == 'hea':
        
        clss_someds.append(get_meta_from_header(open(file).readlines())[-1])

In [17]:
all_snomeds_data = np.unique(np.concatenate(clss_someds).ravel())

In [18]:
# For every snomed id in the dataset, get the corresponding snomed description from remote

baseUrl = 'https://browser.ihtsdotools.org/snowstorm/snomed-ct'
edition = 'MAIN'
version = '2019-07-31'

from urllib.request import urlopen
import json
import re

def getConceptById(snomed_id):
    url = baseUrl + '/browser/' + edition + '/' + version + '/concepts/' + snomed_id
    print(url)
    response = urlopen(url).read()
    data = json.loads(response.decode('utf-8'))

    return data['fsn']['term']

def getDescriptionById(snomed_id):
    url = baseUrl + '/' + edition + '/' + version + '/descriptions/' + snomed_id
    print(url)
    response = urlopen(url).read()
    data = json.loads(response.decode('utf-8'))

    return data['term']

def formatConcept(snomed_desc):
    snomed_desc.lower().replace('(disorder)', '').replace('(finding)', '').strip()
    groups = re.split(r"\((\w+)\)", snomed_desc.replace('observable entity',
                                                     'observable'))[:-1]
    concept = groups[0].lower()
    concept = concept.replace('electrocardiographic:', '')
    concept = concept.replace('electrocardiographic', '')
    concept = concept.replace('on electrocardiogram', '')
    concept = concept.replace('electrocardiogram:', '')
    concept = concept.replace('electrocardiogram', '')
     
    concept = concept.replace('of heart', '')
    concept = concept.lower().strip()
    
    return concept

In [19]:
concepts = {}
for a in all_snomeds_data:
    concepts[str(a)] = formatConcept(getConceptById(str(a)))

https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/368009
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/6374002
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/10370003
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/11157007
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/13640000
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/17338001
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/27885002
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/29320008
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/39732003
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/47665007
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAI

https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/426627000
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/426648003
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/426664006
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/426749004
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/426761007
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/426783006
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/426995002
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/427084000
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/427172004
https://browser.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/2019-07-31/concepts/427393009
https://browser.ihtsdotools.org/snowstorm/snomed-c

In [20]:
df = pd.DataFrame(index=concepts.keys(), columns=['SHORT', 'SNOMED', 'PHYSIONET', 'SCORED', 'SET'])

In [21]:
for key in concepts.keys():
    df.loc[key, 'SNOMED'] = concepts[key]

for entry in dx_map_all[1:]:
    df.loc[entry[1], 'PHYSIONET'] = entry[0]
    df.loc[entry[1], 'SHORT'] = entry[-1]
        
for entry in dx_map_scored[1:]:
    df.loc[entry[1], 'SCORED'] = True
    
for entry in dx_map_unscored[1:]:
    df.loc[entry[1], 'SCORED'] = False

In [22]:
# What is wierd - Now we have classes in the dx map that are nan in the SNOMED
# (classes that are not presend in the dataset)
#index = df['SNOMED'].index[df['SNOMED'].apply(np.isnan)]
for unknown in list(df.loc[pd.isna(df["SNOMED"]), :].index):
    df.loc[unknown, 'SNOMED'] = formatConcept(getConceptById(str(unknown)))

In [23]:
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None):
    df = df.sort_values('SCORED', ascending=False)
    display(df)

,SHORT,SNOMED,PHYSIONET,SCORED,SET
426627000,Brady,bradycardia,bradycardia,True,NaN
59118001,RBBB,right bundle branch block,right bundle branch block,True,NaN
164890007,AFL,atrial flutter,atrial flutter,True,NaN
164889003,AF,atrial fibrillation,atrial fibrillation,True,NaN
164917005,QAb,q wave abnormal,qwave abnormal,True,NaN
426177001,SB,sinus bradycardia,sinus bradycardia,True,NaN
284470004,PAC,premature atrial contraction,premature atrial contraction,True,NaN
111975006,LQT,prolonged qt interval,prolonged qt interval,True,NaN
164934002,TAb,t wave abnormal,t wave abnormal,True,NaN
164947007,LPR,prolonged pr interval,prolonged pr interval,True,NaN


In [24]:
# Check out classes that are note scored
equivalent_classes = [['713427006', '59118001'], ['284470004', '63593006'], ['427172004', '17338001']]

In [25]:
for equ in equivalent_classes:
    print("#" * 50)
    print(df.loc[equ[0]])
    print(df.loc[equ[1]])
    
# By the power nobody appointed me, I decide we go with
# 59118001 (right bundle branch block)
# 284470004 (premature atrial contraction)
# 427172004 (premature ventricular contractions)

##################################################
SHORT                                     CRBBB
SNOMED       complete right bundle branch block
PHYSIONET    complete right bundle branch block
SCORED                                     True
SET                                         NaN
Name: 713427006, dtype: object
SHORT                             RBBB
SNOMED       right bundle branch block
PHYSIONET    right bundle branch block
SCORED                            True
SET                                NaN
Name: 59118001, dtype: object
##################################################
SHORT                                 PAC
SNOMED       premature atrial contraction
PHYSIONET    premature atrial contraction
SCORED                               True
SET                                   NaN
Name: 284470004, dtype: object
SHORT                                    SVPB
SNOMED       supraventricular premature beats
PHYSIONET    supraventricular premature beats
SCORED                 

In [26]:
# New decide if we want to create subsets
df.loc[df['SCORED']]

,SHORT,SNOMED,PHYSIONET,SCORED,SET
426627000,Brady,bradycardia,bradycardia,True,NaN
59118001,RBBB,right bundle branch block,right bundle branch block,True,NaN
164890007,AFL,atrial flutter,atrial flutter,True,NaN
164889003,AF,atrial fibrillation,atrial fibrillation,True,NaN
164917005,QAb,q wave abnormal,qwave abnormal,True,NaN
426177001,SB,sinus bradycardia,sinus bradycardia,True,NaN
284470004,PAC,premature atrial contraction,premature atrial contraction,True,NaN
111975006,LQT,prolonged qt interval,prolonged qt interval,True,NaN
164934002,TAb,t wave abnormal,t wave abnormal,True,NaN
164947007,LPR,prolonged pr interval,prolonged pr interval,True,NaN
